# Exercise 3.3 #

To embed hidden message based on LSB I designed application that:
* only changes minimun nessery data so, add # as an end sign of message and do not alter any data after that to be less invasive on data 
* instead of changing last bit of every byt, change last bite of every 100 byte

#### Load module and original file ####
to read data from audio file use wave module 
load original audio file to soundToEmbed variable 

In [9]:
import wave 
soundToEmbed = wave.open("Ex3_files/Ex3_sound5.wav")

#### Read data of original file ####
*Reusing some code from exercise 3.1 

In [10]:
# extract all the bytes 
extractedBytes = soundToEmbed.readframes(-1)  # data is in bytes object - literal format b'\xfe\xff\...', -1 is to read all frames 

#### Transform data to binary format ####
* from literal hex byte format to int list format
* loop over that int list and change every int to 8 digit binary

In [11]:
# change from literal format to list of integers - took the idea  from https://docs.python.org/3/library/stdtypes.html#bytes
extractedIntList = list(extractedBytes) # data is in a list of integer formal [254, 255, ...]

# change from int to binary 
extractedBinaryList = []
for i in range (len(extractedIntList)):
    extractedBinaryList.append(format(extractedIntList[i], '08b')) # will format from int to binary and append that to list ['11111110','11111111', ...] if numbers are smaller it will fillout begining with 0s to match 8 bit format 

#### Hidden message change to binary format that can be enbeded ####
* take message as string 
* add '#' at the end of that string to indicate the end of the hidden message 
* for every character create binnary representation and store that in a list 

In [2]:
# format hidden, from string to binary sequance  
msg = "Father Christmas does not exist" 
# add '#' at the end to indicate the end of embeded message 
msg = msg + '#'
# turn every character from msg to binary representation and put that in the list
binaryLiteralMsg = ""
for char in msg: # iterate over every character
    binaryLiteralMsg = binaryLiteralMsg + format(ord(char), '08b') # fill with 0, 8 width, b-binary : formating to make sure than that even if int takes less then 8 digit binarry empty space in front is filled with 0 learned from: https://www.programiz.com/python-programming/methods/built-in/format
# binaryLiteralMsg is now one long string filled with binary numbers '01000110...' each character takes 8 digits and comes from: binary<-int<-unicode

#### Embed audio data with hidden message ####
* loop over original audio binary data
* every 100 bytes alter last binary bit by inserting value from binary hidden message 
* exit the loop after altering enough data to fully insert hidden message 

In [13]:
# alter extractedBinaryList by inserting digits from binaryLiteralMsg in the least significant bit every 100 bytes 
index = 0 # to track index of binaryLiteralMsg 
for i in range (100, len(extractedBinaryList), 100):
    temp = list(extractedBinaryList[i]) #change from str to list so I can change the last digit 
    temp[-1] = binaryLiteralMsg[index]
    extractedBinaryList[i] = ''.join(temp)
    index += 1
    if index == len(binaryLiteralMsg):
        break
# extractedBinaryList containes embeded msg at this point 

#### Transform embedded data from binary to bytes object ####
* to write to file change from binary to int 
* from int to byte array 
* from byte array to bytes object where data is hex literal 

In [14]:
# change binarys to int values 
embeddedIntList = []
for i in range(len(extractedBinaryList)):
    embeddedIntList.append(int(extractedBinaryList[i], 2))

# need bytes literal string, first input int list to bytearray then feed bytearray to bytes 
embeddedBytesArray = bytearray(embeddedIntList)
embeddedBytes = bytes(embeddedBytesArray)

#### Create file with embedded hidden message ####
* create file that will store embedded audio
* copy to that file audio parameters from original audio 
* write the data from bytes object to new file - that is the audio data that has hidden data embeded

In [15]:
# create file
embeddedSound = wave.open("Ex3_files/embedded_Ex3_Sound5.wav", 'wb')
#copy parameters from original file
embeddedSound.setparams(soundToEmbed.getparams())
# close resource as soon as done using it
soundToEmbed.close()
# write file with embedded data 
embeddedSound.writeframes(embeddedBytes)
# close resource as soon as done using it 
embeddedSound.close()

#### Testing ####
I decided to use code from task 1 (exercise 3.1) that extracts hidden message, to test if my application correctly emedded the data. 
That requires some code adjustment in extracting binary digits to read last bit every 100 bytes 

In [3]:
#lets test by adjusting code from task 1 

# copied -----------
import wave  # used methods from https://docs.python.org/3/library/wave.html
soundWithMsg = wave.open("embedded_Ex3_sound5.wav", mode='rb')
# extract all the bytes
extractedBytes = soundWithMsg.readframes(-1)  # data is in bytes object - literal format b'\xfe\xff\...', -1 is to read all frames 
soundWithMsg.close() #close the resources as soon as we are done using it 
# change from literal format to list of integers - took the idea  from https://docs.python.org/3/library/stdtypes.html#bytes
extractedIntList = list(extractedBytes) # data is in a list of integer formal [254, 255, ...]
# change from int to binary 
extractedBinaryList = []
for i in range (len(extractedIntList)):
    extractedBinaryList.append(format(extractedIntList[i], 'b')) # will format from int to binary and append that to list ['11111110','11111111', ...]
# extract last bit from every byte which at this step is last binary digit 
lastBitBinaryList = []
# /coppied -----------

# adjusted -------- to go read last bit every 100 bytes and start from 100 bytes 
for i in range (100,len(extractedBinaryList)-800, 800): # iterate over Binary list and step every 8
    joinedLastBits = ""
    for j in range (0,8): # in every step of 8, take last digit from each binary
        joinedLastBits = joinedLastBits + extractedBinaryList[i + j*100][-1] # create string that is created by combingig last digit from 8 consecutive binary list items 
    lastBitBinaryList.append(joinedLastBits) # append that new string to the list 
# /adjusted--------

# coppied----------
# turn lastBitBinaryList to lastBitIntList as a path to later change  from int to ASCII characters 
lastBitIntList = []
for i in range (len(lastBitBinaryList)):
    lastBitIntList.append(int(lastBitBinaryList[i], 2)) # change each binary element to int 
# from int to ASCII character 
decodedMsg = ""
for i in range (len(lastBitIntList)):
    if chr(lastBitIntList[i]) == '#': # after msg the rest would be filled with '#' so might as well stop at that point 
        break
    decodedMsg = decodedMsg + chr(lastBitIntList[i])
print(decodedMsg)
# /coppied-------- 

print(msg.split('#')[0])
if(msg.split('#')[0] == decodedMsg):
    print()
    print('Hidden message is the same as extracted message meaning the application correctly embaddeds data to audio file')

Father Christmas does not exist
Father Christmas does not exist

Hidden message is the same as extracted message meaning the application correctly embaddeds data to audio file
